In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("../data/churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["ChurnBinary"] = df["Churn"].map({"No":0, "Yes":1})

In [ ]:
df["AverageMonthlySpend"] = df["TotalCharges"] / df["tenure"].replace(0,np.nan)

In [ ]:
df[["tenure", "MonthlyCharges", "TotalCharges", "AverageMonthlySpend"]].head()

In [ ]:
df["IsNewCustomer"] = (df["tenure"] <= 12).astype(int)

In [ ]:
df.groupby("IsNewCustomer")["ChurnBinary"].mean()

In [ ]:
df["HasLongTermContract"] = df["Contract"].isin(["One year", "Two year"]).astype(int)

In [ ]:
df.groupby("HasLongTermContract")["ChurnBinary"].mean()

In [ ]:
df["HasSupportServices"] = (
    (df["OnlineSecurity"] == "Yes") |
    (df["TechSupport"] == "Yes")
).astype(int)

In [ ]:
df.groupby("HasSupportServices")["ChurnBinary"].mean()

In [ ]:
monthly_median = df["MonthlyCharges"].median()
df["IsHighMonthlyCharge"] = (df["MonthlyCharges"] > monthly_median).astype(int)

In [ ]:
df.groupby("IsHighMonthlyCharge")["ChurnBinary"].mean()

In [ ]:
def show_churn_rate(df, column):
    churn_rate = df.groupby(column)["ChurnBinary"].mean().sort_values(ascending=False)
    display(churn_rate)
    
    churn_rate.plot(kind="bar")
    plt.title(f"Churn Rate by {column}")
    plt.xlabel(column)
    plt.ylabel("Churn Rate")
    plt.show()

In [ ]:
new_features = [
    "IsNewCustomer",
    "HasLongTermContract",
    "HasSupportServices",
    "IsHighMonthlyCharge"
]

for feature in new_features:
    show_churn_rate(df, feature)

In [ ]:
df["IsHighRiskCustomer"] = (
    (df["IsNewCustomer"] == 1) &
    (df["Contract"] == "Month-to-month") &
    (df["PaymentMethod"] == "Electronic check") &
    (df["IsHighMonthlyCharge"] == 1)
).astype(int)

In [ ]:
df.groupby("IsHighRiskCustomer")["ChurnBinary"].agg(["count", "mean"])

In [ ]:
service_cols = [
    "PhoneService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

In [ ]:
df["TotalServices"] = (df[service_cols] == "Yes").sum(axis=1)

In [ ]:
df.groupby("TotalServices")["ChurnBinary"].agg(["count", "mean"])

In [ ]:
df.groupby("TotalServices")["ChurnBinary"].mean().plot(kind="bar")
plt.title("Churn Rate by Total Services")
plt.xlabel("Total Services")
plt.ylabel("Churn Rate")
plt.show()